# Q1 Flow Matching (Swiss Roll)
In this notebook, we will implement a [Flow Matching](https://arxiv.org/abs/2210.02747) model and train it on the swiss roll dataset.

## Workflow

To start, you will implement the network architectures defined in `flow.py`. You can use this Jupyter Notebook for testing your implementation and visualizing your results. You don't need a serious GPU for these.

To facilitate your testing, either enable the autoreload extension so that your modifications to the `.py` files will be automatically reloaded, or restart your notebook kernel after updating `.py` files. In our testing, we found the autoreload extension somewhat finicky and had to restart the kernel for it to pickup the changed files for some parts.
    


In [ ]:
# Uncomment the following pip command to install requirements
# NOT needed on Delta AI: start Jupyter in the course virtual env instead (see DeltaAI.md), i.e. after
#   module load python/3.10.14; source /work/nvme/bfdu/cs444/fa26/mp5/venv/bin/activate
# ! pip install -r requirements.txt

In [ ]:
# loading autoreload extension for jupyter
%load_ext autoreload
try:
    # IPython 9+: always reload whole modules (its new partial reload can keep running your old code)
    %autoreload 2 --full
except Exception:
    # older IPython (e.g. on Delta AI) has no --full and always reloads whole modules
    %autoreload 2

In [ ]:
import os
import unittest
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

from flow import FlowMatching
from networks import SimpleDecoder, SimpleEncoder

### Swiss Roll Dataset

In [ ]:
def make_swiss_roll(n_samples=100, *, noise=0.0, seed=None):
    generator = torch.Generator()
    generator.manual_seed(seed)
    t = 1.5 * np.pi * (1 + 2 * torch.rand(n_samples, generator=generator))
    x = t * torch.cos(t)
    y = t * torch.sin(t)

    X = torch.stack((x, y), dim=1)
    X += noise * torch.randn(n_samples, 2, generator=generator)
    return X, t

def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
X, t = make_swiss_roll(n_samples=1000, noise=1., seed=42)
plt.figure(figsize=(10, 10))
plt.scatter(X[:, 0], X[:, 1], c=t, cmap=plt.cm.hot)
plt.axis('equal')
plt.show()

## Flow Matching

[Flow Matching](https://arxiv.org/abs/2210.02747) (closely related to [Rectified Flow](https://arxiv.org/abs/2209.03003)) learns a time-dependent **vector field** $u_t^\theta(x)$ that gradually transports noise into data. Sampling then amounts to simulating the *deterministic* ODE $\frac{\mathrm{d}}{\mathrm{d}t} X_t = u_t^\theta(X_t)$ from noise $X_0 \sim \mathcal{N}(0, I_d)$ at $t = 0$ to data at $t = 1$, and using $X_1$ as the sample.

**Convention.** $t = 0$ is pure noise $\epsilon \sim \mathcal{N}(0, I_d)$ and $t = 1$ is data $z$. Be aware that Stable Diffusion 3 and the `diffusers` library use the opposite convention ($t = 0$ is data, $t = 1$ is noise).

**Training.** Pick a data point $z$, a noise sample $\epsilon$ and a time $t \sim \text{Unif}[0, 1]$, and move along the straight line between them:

$$
x_t = t\,z + (1 - t)\,\epsilon .
$$

Along this line, the velocity is $\frac{\mathrm{d}}{\mathrm{d}t} x_t = z - \epsilon$, so we train the network to regress it with the conditional flow matching loss:

$$
\mathcal{L}_{\text{CFM}}(\theta) = \mathbb{E}_{t, z, \epsilon} \left[ \left\| u_t^\theta(x_t) - (z - \epsilon) \right\|^2 \right].
$$

Many pairs $(z, \epsilon)$ pass through the same point $x_t$, so the best the network can do is to predict their *average* velocity. This average velocity field is what transports the noise into the data.

**Time conditioning.** The network must know $t$, since the same point $x$ calls for a different velocity at different times. The provided `get_velocity` method concatenates $t$ to $x$, so for 2D data the network's input size is 3.

**Reuse in Q2.** In Q2 you will train a flow matching model on images by *reusing* your `interpolate`, `get_loss` and `sample` without changes: Q2 only swaps in a different network (`get_velocity`) and a different distribution of training times (`sample_times`). So write these three methods for data of any shape `(N, ...)`, draw the times with `self.sample_times`, and pass the extra arguments `*cond` on to `get_velocity` (in Q1 there are none). The tests below check this.

**Q1.1 [2 pts Autograded]** Implement the `interpolate` method in `flow.py`. This function takes in a batch of data $z$, draws noise $\epsilon \sim \mathcal{N}(0, I_d)$ of the same shape, and draws one time $t \sim \text{Unif}[0, 1]$ for each data point (NOT each pixel/coordinate!), with `self.sample_times`, reshaped to `(N, 1, ..., 1)` (as many dimensions as $z$, i.e. `(N, 1)` for 2D data) so that it broadcasts. Return `eps`, `t` and the interpolated points $x_t$.

Once you have implemented the method, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_flow_interpolate')
unittest.TextTestRunner().run(suite)

**Q1.2 [2 pts Autograded]** Implement the `get_loss` method in `flow.py`. You should first call `interpolate` to obtain $\epsilon$, $t$ and $x_t$, and call `get_velocity` to obtain the predicted velocity $u_t^\theta(x_t)$. Then compute $\mathcal{L}_{\text{CFM}}$ as defined above. The loss should be **averaged** over the batch dimension and the data dimensions (i.e., $\|\cdot\|^2$ divided by $d$).

Once you have implemented the method, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_flow_loss')
unittest.TextTestRunner().run(suite)

**Q1.3 [2 pts Autograded]** Complete the `sample` method in `flow.py`. We simulate the ODE from $t = 0$ to $t = 1$ with the Euler method: with $n$ steps, step size $h = 1/n$ and times $t = 0, h, 2h, \dots, 1 - h$, the update rule is

$$
X_{t+h} = X_t + h\, u_t^\theta(X_t).
$$

The initial noise $X_0 \sim \mathcal{N}(0, I_d)$ is passed in as `x`, and the list of times is provided. Compute the $n$ updates and return the whole trajectory, including the starting point. Note that no noise is injected during sampling: the sampler is deterministic.

Once you have implemented the method, run the following cell to test your implementation.

In [ ]:
suite = unittest.TestLoader().loadTestsFromName('test_functions.TestClass.test_flow_sample')
unittest.TextTestRunner().run(suite)

Now train your flow matching model on the Swiss roll dataset. The network is a small MLP (`SimpleEncoder` followed by `SimpleDecoder`), whose input has one extra dimension for $t$.

In [ ]:
flow = FlowMatching(
    torch.nn.Sequential(
        SimpleEncoder(input_size=2 + 1, hidden_size=128, latent_size=16),
        SimpleDecoder(latent_size=16, hidden_size=128, output_size=2),
    )
)
print(f'Number of parameters in flow matching model: {count_parameters(flow)}')
flow.train()
optimizer = optim.Adam(flow.parameters(), lr=1e-3)
for epoch in range(2000):
    optimizer.zero_grad()
    loss = flow.get_loss(X)
    loss.backward()
    optimizer.step()
    if epoch % 200 == 0:
        print(f'Epoch {epoch}, loss {loss.item():.4f}')
print('Done!')

In [ ]:
n_sample = 1000
flow.eval()
X_gen = flow.sample(torch.randn(n_sample, 2), n_steps=100)[-1]
plt.figure(figsize=(10, 10))
plt.scatter(X[:, 0], X[:, 1], c=t, cmap=plt.cm.hot)
plt.scatter(X_gen[:, 0], X_gen[:, 1], c='b', marker='x')
plt.axis('equal')
plt.show()

### Sampling cost

Flow matching calls the network once per Euler step, so we measure the cost of sampling by the **number of function evaluations (NFE)**, i.e., the number of network calls per sample, which here is simply the number of Euler steps.

The cell below samples from your flow matching model with several NFE budgets, starting from the same noise.

In [ ]:
budgets = [1, 2, 5, 10, 20, 100]
fig, axes = plt.subplots(1, len(budgets), figsize=(6 * len(budgets), 6))
noise = torch.randn(n_sample, 2, generator=torch.Generator().manual_seed(0))
for ax, nfe in zip(axes, budgets):
    X_gen = flow.sample(noise, n_steps=nfe)[-1]
    ax.set_title(f'Flow matching, NFE = {nfe}')
    ax.scatter(X[:, 0], X[:, 1], c=t, cmap=plt.cm.hot)
    ax.scatter(X_gen[:, 0], X_gen[:, 1], c='b', marker='x')
    ax.axis('equal')
plt.show()

**Q1.4 [2 pts Manual graded]** Upload the visualization above and answer the following:
1. Roughly how many NFE are needed before the samples recover the Swiss roll?
2. With a single step (NFE = 1), where do the samples end up, and why? *Hint: what is the best possible prediction of $z - \epsilon$ given only $\epsilon$ at $t = 0$?*

*Each NFE of a large text-to-image model is one pass through a network with billions of parameters, so reducing the number of steps directly reduces generation time. This is one of the reasons recent models such as [Stable Diffusion 3](https://arxiv.org/abs/2403.03206) use flow matching.*